# **MNIST  - CNN for Handwritten Digit Recognition V.2**

#  **1 - Libraries**

In [ ]:
import os 
import numpy as np
import pandas as pd 
import random
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Conv2D, MaxPool2D, Activation, Dropout, BatchNormalization, Flatten
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.image import ImageDataGenerator 
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay


In [ ]:
print("TensorFlow version:", tf.__version__)
print("Num GPUs Available:", len(tf.config.list_physical_devices('GPU')))

# **2 - Data**

## 2.1 Loading Data

In [ ]:
# file path 
train_file_path = "/kaggle/input/digit-recognizer/train.csv"
submission_file_path = "/kaggle/input/digit-recognizer/test.csv"

# loading 
train_data = pd.read_csv(train_file_path)
submission_data = pd.read_csv(submission_file_path)

## 2.2 Data Preprocessing

In [ ]:
y_train_full = train_data["label"]
X_train_full = train_data.drop("label", axis=1)

X_submission = submission_data

In [ ]:
# normalizing 
X_train_full = X_train_full / 225
X_submission = X_submission / 225

# convert the pd DataFrame into np Array 
X_train_full = X_train_full.values.reshape(-1,28,28,1)
X_submission = X_submission.values.reshape(-1,28,28,1)

# one-hot encoding 
y_train_full = to_categorical(y_train_full, num_classes=10)

In [ ]:
print(X_train_full.shape)
print(y_train_full.shape)
print(type(X_train_full))

In [ ]:
# split the full training data set in to Training, Validation and Test set
train_size = 0.8
val_size = 0.1
random_seed = 100 

X_train, X_val_n_test, y_train, y_val_n_test = train_test_split(X_train_full, y_train_full, train_size=train_size, random_state=random_seed)
X_val, X_test, y_val, y_test = train_test_split(X_val_n_test, y_val_n_test, train_size=val_size/(1-train_size), random_state=random_seed)

In [ ]:
print(X_train.shape)
print(X_val.shape)
print(X_test.shape)

## 2.3 Data Augmentation

In [ ]:
# setup Generator 
data_gen = ImageDataGenerator(
    rotation_range = 10, 
    zoom_range = 0.1, 
    width_shift_range = 0.1,
    height_shift_range = 0.1
)

In [ ]:
# Augmented Images preview
a = 8 # a: number of sample image we take for augmentation process
b = 8 # b: number of augmentation we implement on a single sample 

fig, axes = plt.subplots(a, b, figsize=(b,a))

random_indices = random.sample(range(X_train.shape[0]), a)

for i in range(a): 
    for j in range(b): 
        X_sample = X_train[random_indices[i]].reshape((1,28,28,1))
        y_sample = y_train[random_indices[i]].reshape((1,10))

        X_aug, y_aug = next(data_gen.flow(X_sample, y_sample, batch_size=1))

        ax = axes[i][j]
        ax.imshow(X_aug.reshape((28,28)), cmap='gray')
        ax.set_axis_off()

## 2.4 Data Visualization

In [ ]:
fig, axes = plt.subplots(8, 8, figsize=(8,8))
fig.tight_layout(pad=0.01)

m = X_train.shape[0]

for i, ax in enumerate(axes.flat): 
    random_index = np.random.randint(m)

    reshaped_random_X = X_train[random_index].reshape((28,28))

    ax.imshow(reshaped_random_X, cmap='gray')
    ax.set_title(int(np.argmax(y_train[random_index])))
    ax.set_axis_off()


# **3 - CNN Model**

## 3.1 Define the Model

In [ ]:
# Hyperparameters 
batch_size = 64
epochs = 500 
input_shape = X_train.shape[1:]
print(input_shape)

In [ ]:
model_1 = Sequential([
    Conv2D(filters=32, kernel_size=3, input_shape=input_shape, padding='same'),
    BatchNormalization(),
    Activation("relu"),
    MaxPool2D(pool_size=(2,2), strides=2, padding='same'),

    Conv2D(filters=64, kernel_size=3, padding='same'),
    BatchNormalization(),
    Activation("relu"),
    MaxPool2D(pool_size=(2,2), strides=2, padding='same'),

    Conv2D(filters=128, kernel_size=3, padding='same'),
    BatchNormalization(),
    Activation("relu"),
    MaxPool2D(pool_size=(2,2), strides=2, padding='same'),

    Flatten(),

    Dense(units=128, activation='relu'),
    Dropout(0.5),
    Dense(units=10, activation='softmax')  
])

In [ ]:
model_2 = Sequential([
    Conv2D(filters=32, kernel_size=5, input_shape=input_shape, padding='same', activation='relu'),
    Conv2D(filters=32, kernel_size=5, padding='same', activation='relu'),
    MaxPool2D(pool_size=(2,2), strides=2, padding='same'),
    Dropout(0.25),

    Conv2D(filters=64, kernel_size=3, padding='same', activation='relu'),
    Conv2D(filters=64, kernel_size=3, padding='same', activation='relu'),
    MaxPool2D(pool_size=(2,2), strides=2, padding='same'),
    Dropout(0.25),
    
    Flatten(),

    Dense(units=256, activation='relu'),
    Dropout(0.5),
    Dense(units=10, activation='softmax')
    
])

In [ ]:
model_3 = Sequential([
    Conv2D(filters=32, kernel_size=3, activation='relu', input_shape=input_shape),
    BatchNormalization(),
    Conv2D(filters=32, kernel_size=3, activation='relu'),
    BatchNormalization(),
    Conv2D(filters=32, kernel_size=5, strides=2, padding='same', activation='relu'),
    BatchNormalization(),
    Dropout(0.4),
    
    Conv2D(filters=64, kernel_size=3, activation='relu'),
    BatchNormalization(),
    Conv2D(filters=64, kernel_size=3, activation='relu'),
    BatchNormalization(),
    Conv2D(filters=64, kernel_size=5, strides=2, padding='same', activation='relu'),
    BatchNormalization(),
    Dropout(0.4),

    Conv2D(filters=128, kernel_size=4, activation='relu'),
    BatchNormalization(),
    
    Flatten(),

    Dropout(0.4),
    Dense(units=10, activation='softmax')
    
])

In [ ]:
model = model_2

In [ ]:
model.summary()

In [ ]:
model.compile(optimizer='adam', loss="categorical_crossentropy", metrics=["accuracy"])

## 3.2 Fit the Training Data

In [ ]:
early_stopping = keras.callbacks.EarlyStopping(
    patience=5,
    min_delta=0.001,
    restore_best_weights=True
)

In [ ]:
history = model.fit(
    data_gen.flow(X_train, y_train),
    # X_train, y_train,
    validation_data=(X_val, y_val),
    steps_per_epoch=len(X_train)//batch_size,
    epochs=epochs,
    callbacks=[early_stopping]
)

# **4 - Evaluation**

## 4.1 Sample Prediction

In [ ]:
index = 10012

y_pred = model.predict(X_train[index].reshape((1,28,28,1)))
y_pred = int(np.argmax(y_pred))
y_true = int(np.argmax(y_train[index]))
plt.imshow(X_train[index].reshape((28,28)))
plt.title(f"True: {y_true}    |    Pred: {y_pred}")

## 4.2 Evaluation 

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test)
print("Test loss: ", test_loss)
print("Test accuracy: ", test_acc)

# model_2 with augmentation hit 99.317% accuracy on test data set, which is in top 19% of the leaderboard

## 4.3 Loss and Accuracy plot 

In [ ]:
fig, axes = plt.subplots(2, 1)

axes[0].plot(history.history["loss"], color="b", label="Training loss")
axes[0].plot(history.history["val_loss"], color="r", label="Validation loss")
legend = axes[0].legend(loc="best")

axes[1].plot(history.history["accuracy"], color="b", label="Training accuracy")
axes[1].plot(history.history["val_accuracy"], color="r", label="Validation accuracy")
legend = axes[1].legend(loc="best")

## 4.4 Result Display

In [ ]:
# making Predictions

y_test_preds = model.predict(X_test)
y_test_preds = y_test_preds.argmax(axis=1)
y_test_label = y_test.argmax(axis=1)

print(y_test_preds)
print(y_test_label)

In [ ]:
# sorting Prediciton into Valid and Invalid categories
valid_pred_index = []
invalid_pred_index = []

for index, label in enumerate(y_test_preds):
    if label == y_test_label[index]: 
        valid_pred_index.append(index)
    else: 
        invalid_pred_index.append(index)

print(len(valid_pred_index))
print(len(invalid_pred_index))

In [ ]:
def image_display(X, y, y_preds, X_indices, axes, color):
    for i, ax in enumerate(axes.flat):
        image_id = X_indices[i] 
        X_reshape = X[image_id].reshape((28,28))

        ax.imshow(X_reshape, cmap=color)

        ax.set_title("True: {} Pred: {}".format(y[image_id], y_preds[image_id]))
        ax.set_axis_off()

### Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test_label, y_test_preds)

# Display confusion matrix with sklearn
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap=plt.cm.Greens)
plt.title("Confusion Matrix")
plt.show()

### Display some accurate Predictions

In [ ]:
valid_fig, valid_axes = plt.subplots(5, 5, figsize=(15,15))
valid_fig.tight_layout(pad=0.1)

image_display(X_test, y_test_label, y_test_preds, valid_pred_index, valid_axes, "Greens" )

### Display some inaccurate Predictions

In [ ]:
invalid_fig, invalid_axes = plt.subplots(5, 5, figsize=(15,15))
invalid_fig.tight_layout(pad=0.1)

image_display(X_test, y_test_label, y_test_preds, invalid_pred_index, invalid_axes, "Reds" )

# **5 - Submission**

In [ ]:
y_preds = model.predict(X_submission)
y_pred_submission = y_preds.argmax(axis=1)
print(y_pred_submission)

In [ ]:
output_df = pd.DataFrame({
    "ImageId": range(1, len(y_pred_submission)+1),
    "Label": y_pred_submission
})

output_file_path = "/kaggle/working/handwritten_digits_V.2.csv"
output_df.to_csv(output_file_path, index=False)